# End-to-End SLM Agentic Pipeline
This notebook contains the complete pipeline integrated from all separated `.py` files in the repository:

1. `dataset_builder.py`
2. `tool_rl_rewards.py`
3. `rl_phase.py`
4. `trainer.py`
5. `mcp_client.py`
6. `model_handler.py`

## 1. Dataset Builder (`src/data/dataset_builder.py`)

In [1]:
"""
Dataset builder for PHI-3.5 agentic fine-tuning.
Now generates separate train and eval datasets.
"""

import json
import random
from typing import List, Dict, Any, Tuple
from pathlib import Path
from dataclasses import dataclass


@dataclass
class AgenticExample:
    system: str
    instruction: str
    input: str
    output: str
    tools_used: List[str]
    complexity: str


class AgenticDatasetBuilder:
    def __init__(self, output_dir: str = "."):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.available_tools = {
            "web_search": {
                "description": "Search the web for information",
                "parameters": ["query", "max_results"],
            },
            "file_reader": {
                "description": "Read and analyze files",
                "parameters": ["file_path", "operation"],
            },
        }

    def _get_system_prompt(self) -> str:
        tools_str = json.dumps(self.available_tools, indent=2)
        return (
            "You are an AI assistant that can use tools to help solve problems. "
            "You have access to the following tools:\n"
            f"{tools_str}\n\n"
            "To use a tool, respond with the exact XML-like tags:\n"
            "<tool_use>\n"
            "<tool_name>name of tool</tool_name>\n"
            "<parameters>\n"
            "{\"param\": \"value\"}\n"
            "</parameters>\n"
            "</tool_use>"
        )

    # =========================
    # SINGLE STEP
    # =========================
    def create_tool_usage_example(self, tool_name: str, scenario: str) -> List[AgenticExample]:
        system_prompt = self._get_system_prompt()
        original_request = f"Help me with: {scenario}"
        examples = []

        # Step 1: The model should predict the tool call
        thought = f"I need to use {tool_name} to gather information about {scenario}."
        tool_call = self._generate_tool_call(tool_name, scenario)

        examples.append(AgenticExample(
            system=system_prompt,
            instruction=original_request,
            input="",
            output=f"Thought: {thought}\n\nAction:\n{tool_call}",
            tools_used=[tool_name],
            complexity="simple",
        ))

        # Step 2: The model should predict the final answer given the tool result
        observation = self._generate_observation(tool_name, scenario)
        tool_results_text = f"Tool: {tool_name}\nResult: {observation}\n"

        current_instruction = (
            f"Based on the following tool results, provide a comprehensive response:\n\n"
            f"{tool_results_text}\n\nOriginal request: {original_request}"
        )

        final_answer = f"Based on the gathered information, I have completed the task related to {scenario}."

        examples.append(AgenticExample(
            system=system_prompt,
            instruction=current_instruction,
            input="",
            output=f"Final Answer: {final_answer}",
            tools_used=[tool_name],
            complexity="simple",
        ))

        return examples

    # =========================
    # MULTI STEP
    # =========================
    def create_multi_step_example(self, scenario: str, tools: List[str]) -> List[AgenticExample]:
        system_prompt = self._get_system_prompt()
        original_request = f"Help me with this task: {scenario}"
        examples = []

        tool_results_history = []

        for i, tool in enumerate(tools):
            thought = f"Step {i+1}: I should use {tool} to progress on {scenario}."
            tool_call = self._generate_tool_call(tool, scenario)

            # Determine the instruction for the current step
            if i == 0:
                current_instruction = original_request
            else:
                tool_results_text = "\n".join(tool_results_history)
                current_instruction = (
                    f"Based on the following tool results, provide a comprehensive response:\n\n"
                    f"{tool_results_text}\n\nOriginal request: {original_request}"
                )

            examples.append(AgenticExample(
                system=system_prompt,
                instruction=current_instruction,
                input="",
                output=f"Thought: {thought}\n\nAction:\n{tool_call}",
                tools_used=tools,
                complexity="complex",
            ))

            # Simulate the environment observation
            observation = self._generate_observation(tool, scenario)
            tool_results_history.append(f"Tool: {tool}\nResult: {observation}\n")

        # The final step: emitting the final answer
        tool_results_text = "\n".join(tool_results_history)
        current_instruction = (
            f"Based on the following tool results, provide a comprehensive response:\n\n"
            f"{tool_results_text}\n\nOriginal request: {original_request}"
        )
        final_answer = f"I have completed the multi-step task for {scenario} using the available tools."

        examples.append(AgenticExample(
            system=system_prompt,
            instruction=current_instruction,
            input="",
            output=f"Final Answer: {final_answer}",
            tools_used=tools,
            complexity="complex",
        ))

        return examples

    # =========================
    # TOOL CALL
    # =========================
    def _generate_tool_call(self, tool_name: str, scenario: str) -> str:
        params = self._generate_sample_parameters(tool_name, scenario)

        return f"""<tool_use>
<tool_name>{tool_name}</tool_name>
<parameters>
{json.dumps(params, indent=2)}
</parameters>
</tool_use>"""

    # =========================
    # OBSERVATION
    # =========================
    def _generate_observation(self, tool_name: str, scenario: str) -> str:
        if tool_name == "web_search":
            return random.choice([
                f"Found multiple sources discussing {scenario}.",
                f"Search results indicate updates regarding {scenario}.",
                f"Relevant articles highlight insights about {scenario}.",
            ])

        if tool_name == "file_reader":
            return random.choice([
                f"The file contains structured data related to {scenario}.",
                f"Logs indicate patterns related to {scenario}.",
                f"Configuration reveals parameters linked to {scenario}.",
            ])

        return "Tool execution completed."

    # =========================
    # PARAMETERS
    # =========================
    def _generate_sample_parameters(self, tool_name: str, scenario: str) -> Dict[str, Any]:
        return {
            "web_search": lambda s: {
                "query": s,
                "max_results": random.choice([3, 5, 10]),
            },
            "file_reader": lambda s: {
                "file_path": f"/data/{s.replace(' ', '_')}_{random.randint(1,100)}.txt",
                "operation": random.choice(["read", "summarize", "analyze"]),
            },
        }.get(tool_name, lambda s: {})(scenario)

    # =========================
    # SCENARIOS
    # =========================
    def _generate_random_scenario(self, tool_name: str) -> str:
        topics = ["AI", "finance", "healthcare", "sports", "climate change"]
        actions = ["analyze", "compare", "summarize", "investigate"]
        entities = ["India", "USA", "Tokyo", "Bangalore"]

        return f"{random.choice(actions)} {random.choice(topics)} trends in {random.choice(entities)}"

    def _generate_multi_step_scenario(self) -> str:
        tasks = [
            "analyze cost of living between cities",
            "evaluate system performance using logs",
            "compare financial performance across companies",
            "analyze user behavior patterns",
        ]
        return random.choice(tasks)

    # =========================
    # DATASET GENERATION
    # =========================
    def _to_dict(self, example: AgenticExample) -> Dict[str, Any]:

        return {
            "system": example.system,
            "instruction": example.instruction,
            "input": example.input,
            "output": example.output,
            "tools_used": example.tools_used,
            "complexity": example.complexity,
        }
    def generate_dataset(self, num_examples: int = 1000) -> List[Dict[str, Any]]:
        examples = []

        single_tool_count = int(num_examples * 0.6)
        for _ in range(single_tool_count):
            tool = random.choice(list(self.available_tools.keys()))
            scenario = self._generate_random_scenario(tool)
            step_examples = self.create_tool_usage_example(tool, scenario)
            for ex in step_examples:
                examples.append(self._to_dict(ex))

        multi_step_count = num_examples - single_tool_count
        for _ in range(multi_step_count):
            num_tools = random.randint(2, min(3, len(self.available_tools)))
            tools = random.sample(list(self.available_tools.keys()), num_tools)
            scenario = self._generate_multi_step_scenario()
            step_examples = self.create_multi_step_example(scenario, tools)
            for ex in step_examples:
                examples.append(self._to_dict(ex))

        return examples

    # =========================
    # TRAIN / EVAL SPLIT
    # =========================
    def generate_train_eval_split(
        self, num_examples: int = 10000, train_ratio: float = 0.9
    ) -> Tuple[List[Dict], List[Dict]]:
        dataset = self.generate_dataset(num_examples)

        # 🔥 CRITICAL: shuffle before split
        random.shuffle(dataset)

        split_idx = int(len(dataset) * train_ratio)

        train_data = dataset[:split_idx]
        eval_data = dataset[split_idx:]

        return train_data, eval_data

    # =========================
    # SAVE
    # =========================
    def save_dataset(self, examples: List[Dict[str, Any]], filename: str):
        filepath = self.output_dir / filename
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(examples, f, indent=2, ensure_ascii=False)

        print(f"Saved {len(examples)} examples to {filepath}")

    def save_train_eval_datasets(self, train_data, eval_data):
        self.save_dataset(train_data, "train_dataset.json")
        self.save_dataset(eval_data, "eval_dataset.json")

        print("\nDataset split summary:")
        print(f"Train samples: {len(train_data)}")
        print(f"Eval samples: {len(eval_data)}")


# =========================
# MAIN
# =========================
# if False: # __main__ block disabled for notebook
if __name__ == "__main__":
    builder = AgenticDatasetBuilder()

    train_data, eval_data = builder.generate_train_eval_split(
        num_examples=100,
        train_ratio=0.9,
    )

    builder.save_train_eval_datasets(train_data, eval_data)

Saved 216 examples to train_dataset.json
Saved 24 examples to eval_dataset.json

Dataset split summary:
Train samples: 216
Eval samples: 24


## 2. Tool RL Rewards (`src/training/tool_rl_rewards.py`)

In [2]:
"""
Rule-based rewards for agent RL: reasoning traces, tool XML, valid tool names, JSON params.
"""

from __future__ import annotations

import json
import math
import re

ALLOWED_TOOL_NAMES = frozenset({"web_search", "file_reader"})

_TOOL_NAME_RE = re.compile(r"<tool_name>\s*([^<\s]+)\s*</tool_name>", re.IGNORECASE)
_PARAMS_RE = re.compile(r"<parameters>\s*([\s\S]*?)\s*</parameters>", re.IGNORECASE)


def score_tool_completion(text: str, weights: dict[str, float] | None = None) -> float:
    """
    Higher is better. Designed for the step-by-step Agentic schema.
    A valid completion should EITHER be a tool call (Thought + Action) OR a Final Answer.
    """
    w = {
        "thought": 0.20,
        "action": 0.20,
        "tool_use_pair": 0.20,
        "tool_name_ok": 0.15,
        "json_ok": 0.25,
        "final_answer": 0.80,  # High reward for proper final answer format
        "hallucination_penalty": 0.50, # Penalize generating both tool call AND final answer
        "garbage_penalty": 0.15,
    }
    if weights:
        w.update(weights)

    if not text or not text.strip():
        return -1.0

    t = text.strip()
    score = 0.0

    has_thought = re.search(r"\bThought:", t, re.IGNORECASE) or re.search(r"Thought:\s*Step", t, re.IGNORECASE)
    has_action = re.search(r"\bAction:", t, re.IGNORECASE)
    has_tool_pair = "<tool_use>" in t and "</tool_use>" in t
    has_final_answer = re.search(r"Final Answer:", t, re.IGNORECASE)

    if has_thought:
        score += w["thought"]

    # If it's a tool call step
    if has_action and has_tool_pair:
        score += w["action"]
        score += w["tool_use_pair"]

        # Check valid tool names
        m_name = _TOOL_NAME_RE.search(t)
        if m_name and m_name.group(1).lower() in ALLOWED_TOOL_NAMES:
            score += w["tool_name_ok"]

        # Check JSON validity
        m_params = _PARAMS_RE.findall(t)
        json_hits = 0
        for params_str in m_params:
            try:
                obj = json.loads(params_str.strip())
                if isinstance(obj, dict):
                    json_hits += 1
            except (json.JSONDecodeError, TypeError):
                pass

        if json_hits > 0:
            score += w["json_ok"]
        else:
            score -= 0.15

    # If it's a final answer step
    elif has_final_answer:
        score += w["final_answer"]

    # Penalize if it tries to do BOTH (hallucinating observations)
    if has_action and has_final_answer:
        score -= w["hallucination_penalty"]

    # Garbage penalties
    ctrl = sum(1 for c in t if ord(c) < 32 and c not in "\n\r\t")
    if ctrl > 0:
        score -= w["garbage_penalty"] * min(5.0, ctrl / 5.0)

    non_ascii_ratio = sum(1 for c in t if ord(c) > 127) / max(len(t), 1)
    if non_ascii_ratio > 0.35:
        score -= w["garbage_penalty"] * (non_ascii_ratio - 0.35)

    return float(max(-2.0, min(3.0, score)))


def summarize_batch_rewards(rewards: list[float]) -> dict[str, float]:
    if not rewards:
        return {"mean": 0.0, "max": 0.0, "min": 0.0, "std": 0.0}
    n = len(rewards)
    mean = sum(rewards) / n
    var = sum((r - mean) ** 2 for r in rewards) / max(n, 1)
    return {
        "mean": mean,
        "max": max(rewards),
        "min": min(rewards),
        "std": math.sqrt(var),
    }


## 3. RL Phase (`src/training/rl_phase.py`)

In [3]:
"""
Optional post-SFT RL phase (GRPO-style group relative advantages) focused on tool use + reasoning.
Plug-in: enabled via config `rl.enabled`. Does not alter SFT dataset or AgentTrainer layout beyond a hook.
"""

from __future__ import annotations

import json
import logging
import math
import random
from pathlib import Path
from typing import Any

import torch
from datasets import load_dataset



log = logging.getLogger(__name__)


def _patch_dynamic_cache_for_phi3_hub() -> None:
    """Hub Phi-3 vs Transformers 4.48+: restore seen_tokens, get_max_length, get_usable_length on DynamicCache."""
    try:
        from transformers.cache_utils import DynamicCache

        if not hasattr(DynamicCache, "seen_tokens"):
            DynamicCache.seen_tokens = property(lambda self: self.get_seq_length(0))
        if not hasattr(DynamicCache, "get_max_length"):

            def get_max_length(self, layer_idx: int = 0) -> int:
                return self.get_max_cache_shape(layer_idx)

            DynamicCache.get_max_length = get_max_length
        if not hasattr(DynamicCache, "get_usable_length"):

            def get_usable_length(self, new_seq_length: int, layer_idx: int = 0) -> int:
                max_length = self.get_max_cache_shape(layer_idx)
                prev = self.get_seq_length(layer_idx)
                if max_length is not None and max_length > 0 and prev + new_seq_length > max_length:
                    return max_length - new_seq_length
                return prev

            DynamicCache.get_usable_length = get_usable_length
    except Exception as ex:
        log.debug("DynamicCache hub-compat patch skipped: %s", ex)


def _format_prompt_only(tokenizer: Any, system: str, instruction: str, input_text: str | None) -> str:
    user_msg = instruction
    if input_text and input_text.strip():
        user_msg += f"\n\nInput:\n{input_text.strip()}"
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_msg})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def _load_prompt_dataset(config: dict[str, Any], split: str, tokenizer: Any) -> list[str]:
    data = config["data"]
    path = data["train_dataset"] if split == "train" else data["eval_dataset"]
    ds = load_dataset("json", data_files=path)["train"]
    prompts: list[str] = []
    for row in ds:
        system = row.get("system") or ""
        inst = row.get("instruction") or ""
        inp = row.get("input") or ""
        prompts.append(_format_prompt_only(tokenizer, system, inst, inp if inp else None))
    return prompts


def _get_device(model: torch.nn.Module) -> torch.device:
    try:
        p = next(model.parameters())
        return p.device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def completion_log_prob(
    model: torch.nn.Module,
    tokenizer: Any,
    prompt: str,
    completion: str,
    device: torch.device,
) -> torch.Tensor:
    """Sum log pi(completion | prompt) under the model (differentiable)."""
    p_ids = tokenizer(prompt, add_special_tokens=True, return_tensors="pt").input_ids.to(device)
    c_ids = tokenizer(completion, add_special_tokens=False, return_tensors="pt").input_ids.to(device)
    if c_ids.numel() == 0:
        return torch.zeros((), device=device, dtype=torch.float32)

    full = torch.cat([p_ids, c_ids], dim=1)
    out = model(full)
    logits = out.logits[:, :-1, :].float()
    log_probs = torch.log_softmax(logits, dim=-1)
    p_len = p_ids.shape[1]
    c_len = c_ids.shape[1]
    total = torch.zeros((), device=device, dtype=log_probs.dtype)
    for j in range(c_len):
        pos = p_len - 1 + j
        tid = full[0, pos + 1]
        total = total + log_probs[0, pos, tid]
    return total


@torch.no_grad()
def generate_one(
    model: torch.nn.Module,
    tokenizer: Any,
    prompt: str,
    device: torch.device,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
    seed: int,
) -> str:
    if device.type == "cuda":
        g = torch.Generator(device=device)
    else:
        g = torch.Generator()
    g.manual_seed(int(seed) % (2**32))
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(device)
    prompt_len = inputs["input_ids"].shape[1]
    model.eval()
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(0.01, temperature) if temperature > 0 else 1.0,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
        generator=g,
    )
    model.train()
    gen_ids = out[0, prompt_len:]
    return tokenizer.decode(gen_ids, skip_special_tokens=False).strip()


def _rl_eval_pass(
    model: torch.nn.Module,
    tokenizer: Any,
    eval_prompts: list[str],
    cfg: dict[str, Any],
    device: torch.device,
    step: int,
    out_dir: Path,
    num_samples: int,
) -> dict[str, float]:
    model.eval()
    scores: list[float] = []
    sample_text = ""
    max_new = int(cfg.get("max_new_tokens", 384))
    rng = random.Random(int(cfg.get("seed", 42)) + step)
    subset = eval_prompts[: min(len(eval_prompts), num_samples)]
    for i, prompt in enumerate(subset):
        text = generate_one(
            model,
            tokenizer,
            prompt,
            device,
            max_new_tokens=max_new,
            temperature=0.0,
            top_p=1.0,
            seed=rng.randint(0, 2**31 - 1),
        )
        scores.append(score_tool_completion(text))
        if i == 0:
            sample_text = text[:1200]
    model.train()
    stats = summarize_batch_rewards(scores)
    stats["step"] = float(step)
    log.info(
        "[RL eval] step=%s mean_reward=%.4f max=%.4f std=%.4f",
        step,
        stats["mean"],
        stats["max"],
        stats["std"],
    )
    if sample_text:
        log.info("[RL eval] first sample (trunc): %s", sample_text.replace("\n", "\\n")[:500])

    metrics_path = out_dir / "rl_metrics.jsonl"
    with open(metrics_path, "a", encoding="utf-8") as f:
        f.write(json.dumps({**stats, "split": "eval_greedy"}) + "\n")

    return stats


def _save_rl_checkpoint(model: torch.nn.Module, tokenizer: Any, out_dir: Path, step: int) -> None:
    ckpt = out_dir / f"checkpoint-rl-{step}"
    ckpt.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ckpt)
    tokenizer.save_pretrained(ckpt)
    log.info("[RL] saved %s", ckpt.resolve())


def run_rl_phase(config: dict[str, Any], model: torch.nn.Module, tokenizer: Any) -> None:
    """
    GRPO-style updates: per prompt, sample G completions, reward, center advantages within group, policy grad.
    """
    rl = config.get("rl") or {}
    if not rl.get("enabled", False):
        return

    _patch_dynamic_cache_for_phi3_hub()

    device = _get_device(model)
    train_prompts = _load_prompt_dataset(config, "train", tokenizer)
    eval_prompts = _load_prompt_dataset(config, "eval", tokenizer)
    if not train_prompts:
        log.warning("[RL] No training prompts; skipping RL phase.")
        return

    out_root = Path(rl.get("output_dir", "./results_rl"))
    out_root.mkdir(parents=True, exist_ok=True)

    max_steps = int(rl.get("max_steps", 100))
    save_steps = int(rl.get("save_steps", 50))
    eval_steps = int(rl.get("eval_steps", 25))
    num_generations = int(rl.get("num_generations", 4))
    max_new_tokens = int(rl.get("max_new_tokens", 384))
    batch_prompts = int(rl.get("per_device_prompts", 2))
    lr = float(rl.get("learning_rate", 1e-6))
    temperature = float(rl.get("temperature", 0.85))
    top_p = float(rl.get("top_p", 0.95))
    reward_scale = float(rl.get("reward_scale", 1.0))
    max_seq = int(config["data"]["max_seq_length"])
    eval_sample_n = int(rl.get("eval_num_prompts", 12))
    base_seed = int(rl.get("seed", config["data"].get("seed", 42)))

    trainable = [p for p in model.parameters() if p.requires_grad]
    if not trainable:
        log.error("[RL] No trainable parameters; skip RL.")
        return
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=0.0)

    was_gc = bool(getattr(model, "is_gradient_checkpointing", False))
    if was_gc:
        try:
            model.gradient_checkpointing_disable()
        except Exception:
            was_gc = False

    model.train()
    rng = random.Random(base_seed)
    global_step = 0

    log.info(
        "[RL] starting | max_steps=%s G=%s batch_prompts=%s lr=%s out=%s",
        max_steps,
        num_generations,
        batch_prompts,
        lr,
        out_root.resolve(),
    )

    while global_step < max_steps:
        batch = [train_prompts[rng.randrange(len(train_prompts))] for _ in range(batch_prompts)]
        optimizer.zero_grad(set_to_none=True)
        step_losses: list[float] = []
        step_raw_rewards: list[float] = []

        for prompt in batch:
            if len(tokenizer(prompt, add_special_tokens=True)["input_ids"]) > max_seq - max_new_tokens - 8:
                continue

            rewards: list[float] = []
            completions: list[str] = []
            for _g in range(num_generations):
                seed = rng.randint(0, 2**31 - 1)
                with torch.no_grad():
                    comp = generate_one(
                        model,
                        tokenizer,
                        prompt,
                        device,
                        max_new_tokens=max_new_tokens,
                        temperature=temperature,
                        top_p=top_p,
                        seed=seed,
                    )
                completions.append(comp)
                rw = score_tool_completion(comp) * reward_scale
                rewards.append(rw)
                step_raw_rewards.append(rw)

            mean_r = sum(rewards) / len(rewards)
            std_r = math.sqrt(sum((r - mean_r) ** 2 for r in rewards) / max(len(rewards), 1)) + 1e-8
            advantages = [(r - mean_r) / std_r for r in rewards]

            denom = max(1, len(batch) * num_generations)
            for comp, adv in zip(completions, advantages):
                if not comp.strip():
                    continue
                lp = completion_log_prob(model, tokenizer, prompt, comp, device)
                loss = -(float(adv) * lp) / denom
                loss.backward()
                step_losses.append(float(loss.detach().item()))

        if step_losses:
            torch.nn.utils.clip_grad_norm_(trainable, float(rl.get("max_grad_norm", 1.0)))
            optimizer.step()

        global_step += 1
        if step_losses:
            rmean = sum(step_raw_rewards) / max(1, len(step_raw_rewards))
            log.info(
                "[RL] step=%s policy_loss=%.6f | sampled_reward_mean=%.4f (n=%s)",
                global_step,
                sum(step_losses),
                rmean,
                len(step_raw_rewards),
            )
        else:
            log.info("[RL] step=%s (skipped batch; prompts too long)", global_step)

        if global_step % eval_steps == 0:
            _rl_eval_pass(
                model,
                tokenizer,
                eval_prompts,
                rl,
                device,
                global_step,
                out_root,
                eval_sample_n,
            )

        if global_step % save_steps == 0:
            _save_rl_checkpoint(model, tokenizer, out_root, global_step)

    if global_step % save_steps != 0:
        _save_rl_checkpoint(model, tokenizer, out_root, global_step)
    _rl_eval_pass(
        model,
        tokenizer,
        eval_prompts,
        rl,
        device,
        global_step,
        out_root,
        eval_sample_n,
    )

    if was_gc:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

    log.info("[RL] finished. Artifacts under %s", out_root.resolve())


def load_config(path: str) -> dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


## 4. Trainer (`src/training/trainer.py`)

In [ ]:
"""
Phi-3.5 agent fine-tuning (QLoRA) with correct labels, trainable new-token
embeddings, and step-by-step diagnostics.
"""

from __future__ import annotations

import logging
import math
from pathlib import Path
from typing import Any

import torch
import yaml
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)
from transformers.cache_utils import DynamicCache

# Monkeypatch DynamicCache for transformers 5.x compatibility with Phi-3 modeling code
try:
    if not hasattr(DynamicCache, "from_legacy_cache"):
        @classmethod
        def from_legacy_cache(cls, past_key_values):
            cache = cls()
            if past_key_values is not None:
                for layer_idx in range(len(past_key_values)):
                    key_states, value_states = past_key_values[layer_idx]
                    cache.update(key_states, value_states, layer_idx)
            return cache
        DynamicCache.from_legacy_cache = from_legacy_cache

    if not hasattr(DynamicCache, "to_legacy_cache"):
        def to_legacy_cache(self):
            return tuple(self)
        DynamicCache.to_legacy_cache = to_legacy_cache

    if not hasattr(DynamicCache, "seen_tokens"):
        DynamicCache.seen_tokens = property(lambda self: self.get_seq_length(0))

    if not hasattr(DynamicCache, "get_max_length"):
        def get_max_length(self, layer_idx: int = 0) -> int:
            return self.get_max_cache_shape(layer_idx)
        DynamicCache.get_max_length = get_max_length

    if not hasattr(DynamicCache, "get_usable_length"):
        def get_usable_length(self, new_seq_length: int, layer_idx: int = 0) -> int:
            max_length = self.get_max_cache_shape(layer_idx)
            prev = self.get_seq_length(layer_idx)
            if max_length is not None and max_length > 0 and prev + new_seq_length > max_length:
                return max_length - new_seq_length
            return prev
        DynamicCache.get_usable_length = get_usable_length
except Exception as ex:
    pass

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

TOOL_SPECIAL_TOKENS = [
    "<tool_use>",
    "</tool_use>",
    "<tool_name>",
    "</tool_name>",
    "<parameters>",
    "</parameters>",
]


def _format_example(tokenizer: Any, system: str, instruction: str, input_text: str | None, output: str) -> tuple[str, str]:
    """Returns (prompt_prefix, full_text). Loss is applied only on tokens after prompt_prefix."""
    user_msg = instruction
    if input_text:
        user_msg += f"\n\nInput:\n{input_text}"

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_msg})

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Append the actual output, and crucially add EOS token so model learns to stop
    full_text = prompt + output + tokenizer.eos_token
    return prompt, full_text


class AgentDataCollator:
    """Pad input_ids / attention_mask; pad labels with -100 (ignore index)."""

    def __init__(self, tokenizer: Any, pad_to_multiple_of: int | None = 8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        pad_features = [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features]
        batch = self.tokenizer.pad(
            pad_features,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        max_len = batch["input_ids"].size(1)
        labels = torch.full((len(features), max_len), -100, dtype=torch.long)
        for i, f in enumerate(features):
            lab = f["labels"]
            L = min(len(lab), max_len)
            labels[i, :L] = torch.tensor(lab[:L], dtype=torch.long)
        batch["labels"] = labels
        return batch


class MetricsLoggingCallback(TrainerCallback):
    """Log loss, LR, epoch, and perplexity on each reported step."""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        parts = [f"step={state.global_step}"]
        if "loss" in logs and logs["loss"] is not None:
            parts.append(f"train_loss={logs['loss']:.4f}")
            try:
                ppl = math.exp(min(20.0, logs["loss"]))
                parts.append(f"train_ppl={ppl:.2f}")
            except OverflowError:
                parts.append("train_ppl=inf")
        if "learning_rate" in logs:
            parts.append(f"lr={logs['learning_rate']:.2e}")
        if "epoch" in logs:
            parts.append(f"epoch={logs['epoch']:.4f}")
        log.info(" | ".join(parts))

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics or "eval_loss" not in metrics:
            return
        el = float(metrics["eval_loss"])
        try:
            ppl = math.exp(min(20.0, el))
        except OverflowError:
            ppl = float("inf")
        log.info(
            "eval @ step=%s | eval_loss=%.4f | eval_ppl=%.2f",
            state.global_step,
            el,
            ppl,
        )


class AgentTrainer:
    def __init__(self, config_path: str):
        self.config = self._load_config(config_path)
        self.model = None
        self.tokenizer = None
        self.train_dataset = None
        self.eval_dataset = None

    def _load_config(self, path: str) -> dict[str, Any]:
        with open(path, "r", encoding="utf-8") as f:
            return yaml.safe_load(f)

    def setup_model_and_tokenizer(self) -> None:
        model_id = self.config["model"]["name"]
        log.info("Loading tokenizer (%s)...", model_id)

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            trust_remote_code=True,
            padding_side="right",
        )

        added = self.tokenizer.add_special_tokens(
            {"additional_special_tokens": TOOL_SPECIAL_TOKENS}
        )
        log.info("Added %d tool special tokens (expected %d).", added, len(TOOL_SPECIAL_TOKENS))

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        qcfg = self.config.get("quantization", {})
        use_4bit = bool(qcfg.get("load_in_4bit", True))
        quant_config = None
        if use_4bit:
            compute_dtype = torch.float16
            bnb_dtype = qcfg.get("bnb_4bit_compute_dtype", "float16")
            if isinstance(bnb_dtype, str) and "bfloat16" in bnb_dtype.lower():
                compute_dtype = torch.bfloat16
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=compute_dtype,
                bnb_4bit_quant_type=qcfg.get("bnb_4bit_quant_type", "nf4"),
                bnb_4bit_use_double_quant=bool(qcfg.get("bnb_4bit_use_double_quant", False)),
            )
            log.info("Using 4-bit quantization (QLoRA).")

        log.info("Loading model...")
        torch_dtype = torch.bfloat16 if self.config["training"].get("bf16") else torch.float16
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            trust_remote_code=True,
            device_map="auto",
            quantization_config=quant_config,
            torch_dtype=torch_dtype if not use_4bit else None,
        )
        self.model.resize_token_embeddings(len(self.tokenizer))

        if use_4bit:
            self.model = prepare_model_for_kbit_training(self.model)

        lcfg = self.config["lora"]
        modules_to_save = lcfg.get("modules_to_save") or ["embed_tokens", "lm_head"]
        lora_config = LoraConfig(
            r=lcfg["r"],
            lora_alpha=lcfg["lora_alpha"],
            target_modules=lcfg["target_modules"],
            lora_dropout=lcfg["lora_dropout"],
            bias="none",
            task_type=TaskType.CAUSAL_LM,
            modules_to_save=modules_to_save,
        )
        self.model = get_peft_model(self.model, lora_config)
        self.model.print_trainable_parameters()

        tcfg = self.config["training"]
        if tcfg.get("gradient_checkpointing", True):
            self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

    def _tokenize_batch(self, examples: dict[str, list]) -> dict[str, Any]:
        max_len = int(self.config["data"]["max_seq_length"])
        prompts: list[str] = []
        full_texts: list[str] = []

        for system, instruction, input_text, output in zip(
            examples.get("system", [""] * len(examples["instruction"])),
            examples["instruction"],
            examples["input"],
            examples["output"],
        ):
            inp = (input_text or "").strip()
            prompt, full = _format_example(self.tokenizer, system, instruction, inp if inp else None, output)
            prompts.append(prompt)
            full_texts.append(full)

        enc = self.tokenizer(
            full_texts,
            truncation=True,
            max_length=max_len,
            padding=False,
            return_offsets_mapping=True,
            add_special_tokens=True,
        )

        all_input_ids: list[list[int]] = []
        all_attention: list[list[int]] = []
        all_labels: list[list[int]] = []

        for i, prompt in enumerate(prompts):
            cut = len(prompt)
            ids = enc["input_ids"][i]
            offsets = enc["offset_mapping"][i]
            labels: list[int] = []
            for tid, (start, _end) in zip(ids, offsets):
                if start >= cut:
                    labels.append(int(tid))
                else:
                    labels.append(-100)
            all_input_ids.append(ids)
            all_attention.append(enc["attention_mask"][i])
            all_labels.append(labels)

        return {"input_ids": all_input_ids, "attention_mask": all_attention, "labels": all_labels}

    def prepare_datasets(self) -> None:
        data = self.config["data"]
        train_path = data["train_dataset"]
        eval_path = data["eval_dataset"]
        log.info("Loading JSON datasets...")

        train_ds = load_dataset("json", data_files=train_path)["train"]
        eval_ds = load_dataset("json", data_files=eval_path)["train"]

        self.train_dataset = train_ds.map(
            self._tokenize_batch,
            batched=True,
            batch_size=32,
            remove_columns=train_ds.column_names,
            desc="Tokenizing train",
        )
        self.eval_dataset = eval_ds.map(
            self._tokenize_batch,
            batched=True,
            batch_size=32,
            remove_columns=eval_ds.column_names,
            desc="Tokenizing eval",
        )
        log.info("Train rows: %d | Eval rows: %d", len(self.train_dataset), len(self.eval_dataset))

    def _response_label_stats(self, split: str, n: int = 512) -> None:
        ds = self.train_dataset if split == "train" else self.eval_dataset
        total_lab = 0
        masked = 0
        end = min(n, len(ds))
        for i in range(end):
            for x in ds[i]["labels"]:
                total_lab += 1
                if x == -100:
                    masked += 1
        log.info(
            "[%s] label positions (first %d rows): %.1f%% masked (instruction/padding)",
            split,
            end,
            100.0 * masked / max(1, total_lab),
        )

    def verify_setup(self) -> None:
        """Sanity checks: special token ids, one forward pass, decoded sample."""
        assert self.model is not None and self.tokenizer is not None
        for t in TOOL_SPECIAL_TOKENS:
            tid = self.tokenizer.convert_tokens_to_ids(t)
            if tid is None or tid < 0:
                log.warning("Token %s missing from vocab.", t)
            else:
                log.info("Token %s -> id %s", t, tid)

        self._response_label_stats("train")
        self._response_label_stats("eval")

        collator = AgentDataCollator(self.tokenizer)
        batch = collator([self.train_dataset[0], self.train_dataset[1]])
        self.model.eval()
        device = next(self.model.parameters()).device
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            out = self.model(**batch)
        loss = float(out.loss.item())
        log.info("Sanity forward on 2 examples: loss=%.4f (finite=%s)", loss, math.isfinite(loss))

        ids = self.train_dataset[0]["input_ids"]
        lab = self.train_dataset[0]["labels"]
        vis_ids = [tid for tid, lb in zip(ids, lab) if lb != -100][:80]
        snippet = self.tokenizer.decode(vis_ids, skip_special_tokens=False)
        log.info("First example response-prefix decode (truncated): %s", snippet[:400].replace("\n", "\\n"))

    def setup_training_args(self) -> TrainingArguments:
        t = self.config["training"]
        ev = self.config.get("evaluation", {})
        fp16 = bool(t.get("fp16", False))
        bf16 = bool(t.get("bf16", True))

        report = t.get("report_to", "none")
        if report in (None, "", "none"):
            report_list: list[str] = []
        elif isinstance(report, str):
            report_list = [report]
        else:
            report_list = list(report)

        return TrainingArguments(
            output_dir=t["output_dir"],
            num_train_epochs=float(t["num_train_epochs"]),
            max_steps=int(t.get("max_steps", -1)),
            per_device_train_batch_size=int(t["per_device_train_batch_size"]),
            per_device_eval_batch_size=int(t["per_device_eval_batch_size"]),
            gradient_accumulation_steps=int(t["gradient_accumulation_steps"]),
            learning_rate=float(t["learning_rate"]),
            weight_decay=float(t.get("weight_decay", 0.0)),
            max_grad_norm=float(t.get("max_grad_norm", 1.0)),
            lr_scheduler_type=str(t.get("lr_scheduler_type", "cosine")),
            warmup_ratio=float(t.get("warmup_ratio", 0.03)),
            logging_steps=int(t.get("logging_steps", 10)),
            save_steps=int(t.get("save_steps", 500)),
            eval_strategy=str(ev.get("evaluation_strategy", "steps")),
            eval_steps=int(ev.get("eval_steps", t.get("save_steps", 500))),
            save_strategy=str(ev.get("save_strategy", "steps")),
            load_best_model_at_end=bool(ev.get("load_best_model_at_end", True)),
            metric_for_best_model=str(ev.get("metric_for_best_model", "eval_loss")),
            greater_is_better=bool(ev.get("greater_is_better", False)),
            fp16=fp16 and not bf16,
            bf16=bf16,
            gradient_checkpointing=bool(t.get("gradient_checkpointing", True)),
            optim=str(t.get("optim", "adamw_torch")),
            remove_unused_columns=False,
            report_to=report_list,
            save_total_limit=int(t.get("save_total_limit", 3)),
            logging_first_step=True,
            prediction_loss_only=True,
        )

    def train(self) -> Trainer:
        self.setup_model_and_tokenizer()
        self.prepare_datasets()
        self.verify_setup()

        args = self.setup_training_args()
        collator = AgentDataCollator(
            self.tokenizer,
            pad_to_multiple_of=int(self.config["training"].get("pad_to_multiple_of", 8)),
        )

        trainer = Trainer(
            model=self.model,
            args=args,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            processing_class=self.tokenizer,
            data_collator=collator,
            callbacks=[MetricsLoggingCallback()],
        )

        log.info("Starting training...")
        trainer.train()

        out_dir = Path(self.config["training"]["output_dir"])
        out_dir.mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(out_dir)
        self.tokenizer.save_pretrained(out_dir)
        log.info("Saved adapter + tokenizer to %s", out_dir.resolve())

        self._maybe_run_rl_phase()

        return trainer

    def _maybe_run_rl_phase(self) -> None:
        rl_cfg = self.config.get("rl") or {}
        if not rl_cfg.get("enabled", False):
            return
        import sys


        import rl_phase

        run_rl_phase(self.config, self.model, self.tokenizer)


# if False: # __main__ block disabled for notebook
if __name__ == "__main__":
    # cfg = Path(__file__).resolve().parents[2] / "config" / "training_config.yaml"
    cfg = "/content/training_config.yaml"
    AgentTrainer(str(cfg)).train()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


trainable params: 205,625,344 || all params: 4,026,416,128 || trainable%: 5.1069


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Tokenizing train:   0%|          | 0/216 [00:00<?, ? examples/s]

Tokenizing eval:   0%|          | 0/24 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss


In [5]:
pip install -U bitsandbytes>=0.46.1

## 5. MCP Client (`src/inference/mcp_client.py`)

In [ ]:
"""
MCP (Model Context Protocol) client for interacting with external tools and servers.
Refactored to use the official MCP SDK with SSE (Server-Sent Events) transport.
"""

import json
import asyncio
import logging
from typing import Any, Optional
from dataclasses import dataclass

from mcp import ClientSession
from mcp.client.sse import sse_client
from pydantic import BaseModel


@dataclass
class ToolCall:
    """Represents a tool call extracted from model output."""

    name: str
    parameters: dict[str, Any]
    call_id: Optional[str] = None


class MCPServerConfig(BaseModel):
    """Configuration for an MCP server."""

    name: str
    url: str
    api_key: Optional[str] = None
    timeout: int = 30
    headers: Optional[dict[str, str]] = None


class MCPClient:
    """Client for interacting with MCP servers via SSE transport and managing tool calls."""

    def __init__(self, server_configs: list[MCPServerConfig]):
        """Initialize MCP client with server configurations."""
        self.server_configs = {config.name: config for config in server_configs}
        self.logger = logging.getLogger(__name__)

        # Active sessions and their tools
        self.sessions: dict[str, ClientSession] = {}
        self.available_tools: dict[str, dict[str, Any]] = {}
        self.tool_to_server: dict[str, str] = {}

    async def connect_to_server(self, server_name: str) -> None:
        """Connect to an MCP server via SSE and initialize the session."""
        if server_name not in self.server_configs:
            raise ValueError(f"Server {server_name} not configured")

        if server_name in self.sessions:
            self.logger.info(f"Already connected to {server_name}")
            return

        config: MCPServerConfig = self.server_configs[server_name]

        try:
            # Prepare headers
            headers = config.headers.copy() if config.headers else {}
            if config.api_key:
                headers["Authorization"] = f"Bearer {config.api_key}"

            # Connect to the server using SSE transport
            read_stream, write_stream = await sse_client(
                url=config.url, headers=headers, timeout=config.timeout
            )

            # Create and initialize session
            session = ClientSession(read_stream, write_stream)
            await session.initialize()

            self.sessions[server_name] = session

            # Discover tools from this server
            await self._discover_tools_from_server(server_name, session)

            self.logger.info(f"Connected to MCP server via SSE: {server_name}")

        except Exception as e:
            self.logger.error(f"Error connecting to {server_name}: {e}")
            raise

    async def _discover_tools_from_server(
        self, server_name: str, session: ClientSession
    ) -> None:
        """Discover available tools from a connected MCP server."""
        try:
            # List available tools from the server
            tools_response = await session.list_tools()

            for tool in tools_response.tools:
                tool_info = {
                    "server": server_name,
                    "name": tool.name,
                    "description": tool.description or "",
                    "input_schema": tool.inputSchema,
                }

                self.available_tools[tool.name] = tool_info
                self.tool_to_server[tool.name] = server_name

                self.logger.info(
                    f"Registered tool '{tool.name}' from server '{server_name}'"
                )

        except Exception as e:
            self.logger.error(f"Error discovering tools from {server_name}: {e}")
            raise

    async def connect_all_servers(self) -> None:
        """Connect to all configured MCP servers."""
        tasks = [
            self.connect_to_server(server_name)
            for server_name in self.server_configs.keys()
        ]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        # Log any connection failures
        for server_name, result in zip(self.server_configs.keys(), results):
            if isinstance(result, Exception):
                self.logger.error(f"Failed to connect to {server_name}: {result}")

    def parse_tool_calls(self, model_output: str) -> list[ToolCall]:
        """Parse tool calls from model output."""
        tool_calls = []

        # Look for tool usage blocks in the output
        lines = model_output.split("\n")
        i = 0

        while i < len(lines):
            line = lines[i].strip()

            if line == "<tool_use>":
                # Found start of tool use block
                tool_call = self._parse_single_tool_call(lines, i)
                if tool_call:
                    tool_calls.append(tool_call)

                # Skip to end of this tool block
                while i < len(lines) and lines[i].strip() != "</tool_use>":
                    i += 1
            i += 1

        return tool_calls

    def _parse_single_tool_call(
        self, lines: list[str], start_idx: int
    ) -> Optional[ToolCall]:
        """Parse a single tool call block."""
        tool_name = None
        parameters = {}
        call_id = None

        i = start_idx + 1
        while i < len(lines):
            line = lines[i].strip()

            if line == "</tool_use>":
                break
            elif line == "<tool_name>":
                # Get tool name
                i += 1
                if i < len(lines):
                    tool_name = lines[i].strip()
            elif line == "<call_id>":
                # Get call ID if present
                i += 1
                if i < len(lines):
                    call_id = lines[i].strip()
            elif line == "<parameters>":
                # Parse parameters JSON
                i += 1
                param_lines = []
                while i < len(lines) and lines[i].strip() != "</parameters>":
                    param_lines.append(lines[i])
                    i += 1

                try:
                    param_text = "\n".join(param_lines)
                    parameters = json.loads(param_text)
                except json.JSONDecodeError as e:
                    self.logger.error(f"Error parsing parameters: {e}")
                    return None

            i += 1

        if tool_name and tool_name in self.available_tools:
            return ToolCall(name=tool_name, parameters=parameters, call_id=call_id)

        return None

    async def execute_tool_call(self, tool_call: ToolCall) -> dict[str, Any]:
        """Execute a tool call on the appropriate MCP server."""
        if tool_call.name not in self.available_tools:
            return {"error": f"Unknown tool: {tool_call.name}", "success": False}

        server_name = self.tool_to_server.get(tool_call.name)
        if not server_name or server_name not in self.sessions:
            return {
                "error": f"Server for tool {tool_call.name} not connected",
                "success": False,
            }

        session = self.sessions[server_name]

        try:
            # Call the tool
            result = await session.call_tool(
                tool_call.name, arguments=tool_call.parameters
            )

            # Extract content from the result
            content = self._extract_tool_result_content(result)

            return {
                "result": content,
                "success": not result.isError,
                "tool": tool_call.name,
                "call_id": tool_call.call_id,
            }

        except Exception as e:
            self.logger.error(f"Error executing {tool_call.name}: {e}")
            return {
                "error": str(e),
                "success": False,
                "tool": tool_call.name,
                "call_id": tool_call.call_id,
            }

    def _extract_tool_result_content(self, result) -> Any:
        """Extract content from CallToolResult."""
        if not result.content:
            return None

        # Handle multiple content items
        if len(result.content) == 1:
            content_item = result.content[0]
            if hasattr(content_item, "text"):
                try:
                    return json.loads(content_item.text)
                except json.JSONDecodeError:
                    return content_item.text
            return content_item
        else:
            extracted = []
            for item in result.content:
                if hasattr(item, "text"):
                    try:
                        extracted.append(json.loads(item.text))
                    except json.JSONDecodeError:
                        extracted.append(item.text)
                else:
                    extracted.append(item)
            return extracted

    async def execute_tool_calls(
        self, tool_calls: list[ToolCall]
    ) -> list[dict[str, Any]]:
        """Execute multiple tool calls concurrently."""
        tasks = [self.execute_tool_call(call) for call in tool_calls]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        processed_results = []
        for i, result in enumerate(results):
            if isinstance(result, Exception):
                processed_results.append(
                    {
                        "error": str(result),
                        "success": False,
                        "tool": tool_calls[i].name,
                        "call_id": tool_calls[i].call_id,
                    }
                )
            else:
                processed_results.append(result)

        return processed_results

    async def reconnect_server(self, server_name: str) -> None:
        """Reconnect to a specific server if connection is lost."""
        if server_name in self.sessions:
            try:
                await self.sessions[server_name].close()
            except Exception as e:
                self.logger.warning(f"Error closing old session for {server_name}: {e}")

            del self.sessions[server_name]

        await self.connect_to_server(server_name)

    async def close(self):
        """Close all MCP server connections."""
        for server_name, session in self.sessions.items():
            try:
                await session.close()
                self.logger.info(f"Closed connection to {server_name}")
            except Exception as e:
                self.logger.error(f"Error closing {server_name}: {e}")

        self.sessions.clear()
        self.available_tools.clear()
        self.tool_to_server.clear()

    def get_available_tools_description(self) -> str:
        """Get a description of all available tools for the model."""
        if not self.available_tools:
            return "No tools available. Please connect to MCP servers first."

        descriptions = []
        descriptions.append("Available tools:")

        for tool_name, config in self.available_tools.items():
            descriptions.append(f"\n- {tool_name}: {config['description']}")

            # Show input schema if available
            schema = config.get("input_schema", {})
            if schema and "properties" in schema:
                params = []
                required = schema.get("required", [])
                for param_name, param_info in schema["properties"].items():
                    param_type = param_info.get("type", "any")
                    is_required = "*" if param_name in required else ""
                    params.append(f"{param_name}({param_type}{is_required})")

                if params:
                    descriptions.append(f"  Parameters: {', '.join(params)}")

        descriptions.append("\n\nTo use a tool, format your response like:")
        descriptions.append("<tool_use>")
        descriptions.append("<tool_name>tool_name</tool_name>")
        descriptions.append("<parameters>")
        descriptions.append('{"param1": "value1", "param2": "value2"}')
        descriptions.append("</parameters>")
        descriptions.append("</tool_use>")

        return "\n".join(descriptions)

    def get_tools_for_prompt(self) -> list[dict[str, Any]]:
        """Get tools in a format suitable for model prompts."""
        tools = []
        for tool_name, config in self.available_tools.items():
            tools.append(
                {
                    "name": tool_name,
                    "description": config["description"],
                    "input_schema": config["input_schema"],
                }
            )
        return tools

    async def list_resources(self, server_name: str) -> list[dict[str, Any]]:
        """List available resources from a specific server."""
        if server_name not in self.sessions:
            raise ValueError(f"Not connected to server: {server_name}")

        session = self.sessions[server_name]
        try:
            resources_response = await session.list_resources()
            return [
                {
                    "uri": resource.uri,
                    "name": resource.name,
                    "description": resource.description,
                    "mime_type": resource.mimeType,
                }
                for resource in resources_response.resources
            ]
        except Exception as e:
            self.logger.error(f"Error listing resources from {server_name}: {e}")
            return []

    async def read_resource(self, server_name: str, uri: str) -> dict[str, Any]:
        """Read a resource from a specific server."""
        if server_name not in self.sessions:
            raise ValueError(f"Not connected to server: {server_name}")

        session = self.sessions[server_name]
        try:
            resource_response = await session.read_resource(uri)
            return {
                "uri": uri,
                "contents": resource_response.contents,
                "success": True,
            }
        except Exception as e:
            self.logger.error(f"Error reading resource {uri} from {server_name}: {e}")
            return {
                "uri": uri,
                "error": str(e),
                "success": False,
            }


## 6. Model Handler (`src/inference/model_handler.py`)

In [ ]:
"""
Model handler for fine-tuned agent inference.
"""

import os
import json
import logging
import asyncio
from typing import Optional, Any
from dataclasses import dataclass

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, pipeline



logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


@dataclass
class GenerationParams:
    """Parameters for text generation."""

    max_new_tokens: int = 512
    temperature: float = 0.7
    top_p: float = 0.9
    top_k: int = 50
    do_sample: bool = True
    repetition_penalty: float = 1.1
    pad_token_id: Optional[int] = None


class AgentModelHandler:
    """
    AgentModelHandler is a class for managing the lifecycle and inference of a language model agent,
    optionally with LoRA adapters and tool integration via an MCP client.

    This handler is responsible for:
    - Loading a base language model and tokenizer from a specified path.
    - Optionally loading a LoRA adapter for fine-tuned weights.
    - Managing device placement, torch dtype, and trust settings.
    - Formatting prompts for chat and instruction-following tasks, including chat history.
    - Generating responses, with support for tool usage via an MCP client, including iterative tool calls.
    - Providing both synchronous and asynchronous batch generation interfaces.
    - Returning detailed model and environment information for diagnostics.

    Attributes:
        base_model_path (str): Path to the base model directory.
        adapter_path (str, optional): Path to the LoRA adapter directory.
        mcp_client (MCPClient): Client for tool integration and tool call execution.
        device (str): Device mapping for model placement (e.g., "cpu", "cuda", "auto").
        torch_dtype (str): Torch data type for model weights (e.g., "float16", "auto").
        trust_remote_code (bool): Whether to trust remote code execution for model/tokenizer.
        padding_side (str): Padding side for the tokenizer ("left" or "right").
        attn_implementation (str): Attention implementation for model loading.
        sys_prompt (str): System prompt loaded from file, used as context for all generations.
        tokenizer (AutoTokenizer): Tokenizer instance for the model.
        model (AutoModelForCausalLM or PeftModel): Loaded model instance.
        generation_config (GenerationConfig): Default generation configuration.
        pipeline (transformers.Pipeline): Text generation pipeline for easier inference.
        logger (logging.Logger): Logger for status and error reporting.

    Methods:
        __init__(...): Initialize the handler, load model/tokenizer, and set up logging.
        _load_model(): Internal method to load model, tokenizer, adapter, and system prompt.
        _initialized(): Check if model and tokenizer are loaded.
        format_prompt(instruction, input_text="", chat_history=None): Format a prompt for the model.
        async chat(): Interactive chat loop for user input and model response.
        async generate_response(...): Generate a response, optionally using tools via MCP client.
        async _generate_text(...): Generate text from the model given an instruction and input.
        _format_tool_results(tool_results): Format tool results for prompt inclusion.
        async batch_generate_async(...): Asynchronously generate responses for a batch of instructions.
        batch_generate(...): Synchronous wrapper for batch generation.
        get_model_info(): Return a dictionary with model and environment information.
        __repr__(): String representation of the handler instance.
    """

    def __init__(
        self,
        base_model_path: str,
        mcp_client: MCPClient,
        adapter_path: str = None,
        device: str = "auto",
        torch_dtype: str = "auto",
        trust_remote_code: bool = False,
        random_seed: int = 42,
        padding_side: str = "left",
        attn_implementation: str = "eager",
    ) -> None:
        """
        Initialize the agent model handler.

        Args:
            base_model_path: Path to the base model
            adapter_path: Path to the LoRA adapter
            mcp_client: MCP client for tool integration
            device: Device mapping for model placement
            torch_dtype: Torch data type for model weights
            trust_remote_code: Whether to trust remote code execution
            random_seed: Random seed for reproducibility
            padding_side: Padding side for tokenizer
        """
        self.base_model_path = base_model_path
        self.adapter_path = adapter_path
        self.mcp_client = mcp_client
        self.device = device
        self.torch_dtype = torch_dtype
        self.trust_remote_code = trust_remote_code
        self.padding_side = padding_side
        self.attn_implementation = attn_implementation
        self.sys_prompt: str = None

        # Set random seed for reproducibility
        torch.random.manual_seed(random_seed)

        # Initialize components
        self.tokenizer = None
        self.model = None
        self.sys_prompt: str = None
        self.generation_config: GenerationConfig = None
        self.pipeline: pipeline = None

        # Setup logging
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)

        # Load model and tokenizer
        self._load_model()

    def _load_model(self) -> None:
        """Load the fine-tuned model and tokenizer."""
        try:
            self.logger.info(f"Loading tokenizer from: {self.base_model_path}")

            # Load tokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.base_model_path,
                trust_remote_code=self.trust_remote_code,
                padding_side=self.padding_side,
            )

            # Set pad token if not available
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            self.logger.info(f"Loading base model from: {self.base_model_path}")

            # Prepare model loading arguments
            model_args = {
                "dtype": self.torch_dtype,
                "device_map": self.device,
                "trust_remote_code": self.trust_remote_code,
                "attn_implementation": self.attn_implementation,
            }

            # Load base model
            base_model = AutoModelForCausalLM.from_pretrained(
                self.base_model_path, **model_args
            )

            # Load LoRA adapter for fine-tune models if available, otherwise continue with base model
            if self.adapter_path:
                self.logger.info(f"Loading LoRA adapter from: {self.base_model_path}")
                self.model = PeftModel.from_pretrained(base_model, self.adapter_path)
            else:
                self.logger.info("Using base model")
                self.model = base_model

            # Set the model to evaluation mode
            self.model.eval()

            # Set up generation configuration with better defaults
            self.generation_config = GenerationConfig(
                max_new_tokens=2048,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                repetition_penalty=1.1,
            )

            # Generate system prompt dynamically matching training format
            tools_str = "{}"
            try:
                if self.mcp_client and hasattr(self.mcp_client, 'available_tools'):
                    tools_str = json.dumps(self.mcp_client.available_tools, indent=2)
            except Exception as e:
                self.logger.warning(f"Could not load tools for sys_prompt: {e}")

            self.sys_prompt = (
                "You are an AI assistant that can use tools to help solve problems. "
                "You have access to the following tools:\n"
                f"{tools_str}\n\n"
                "To use a tool, respond with the exact XML-like tags:\n"
                "<tool_use>\n"
                "<tool_name>name of tool</tool_name>\n"
                "<parameters>\n"
                "{\"param\": \"value\"}\n"
                "</parameters>\n"
                "</tool_use>"
            )

            # Initialize pipeline for easier generation
            self.pipeline = pipeline(
                "text-generation",
                model=self.model,
                tokenizer=self.tokenizer,
            )

            self.logger.info("Model loaded successfully!")
        except Exception as e:
            self.logger.error(f"Failed to load model: {e}")
            raise

    def _initialized(self) -> bool:
        """Check if the model and tokenizer are properly initialized."""
        return self.model is not None and self.tokenizer is not None

    def format_prompt(
        self,
        instruction: str,
        input_text: str = "",
        chat_history: list[dict[str, str]] = None,
    ) -> str:
        """
        Format the prompt for the fine-tuned model.

        Args:
            instruction: The main instruction/query
            input_text: Optional additional input context

        Returns:
            Formatted prompt string
        """
        messages = []
        if self.sys_prompt:
            messages.append({"role": "system", "content": self.sys_prompt})

        if chat_history:
            for h in chat_history[-50:]:  # Keep history manageable
                messages.append({"role": "user", "content": h["user"]})
                messages.append({"role": "assistant", "content": h["assistant"]})

        user_msg = instruction
        if input_text:
            user_msg += f"\n\nInput:\n{input_text}"

        messages.append({"role": "user", "content": user_msg})
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    async def chat(self) -> None:
        """
        Handle a chat message from the user.

        Args:
            user_message: The message sent by the user

        Returns:
            The assistant's response
        """
        while True:
            message = input("> ")
            if message.lower() in ["exit", "quit", "q"]:
                return

            response = await self.generate_response(
                instruction=message, generation_params=self.generation_config
            )
            print(f"🤖: {response['final_response']}\n")

    async def generate_response(
        self,
        instruction: str,
        input_text: str = "",
        generation_params: Optional[GenerationParams] = None,
        max_tool_iterations: int = 3,
        use_pipeline: bool = True,
    ) -> dict[str, Any]:
        """
        Generate response with tool usage capability.

        Args:
            instruction: The main instruction/query
            input_text: Optional additional input context
            generation_params: Generation parameters
            max_tool_iterations: Maximum number of tool usage iterations
            use_pipeline: Whether to use the transformers pipeline for generation

        Returns:
            dictionary containing the response and metadata
        """
        if not self._initialized():
            raise RuntimeError("Model components not properly initialized")

        if generation_params is None:
            generation_params = GenerationParams()

        conversation_history = []
        current_instruction = instruction
        current_input = input_text

        for iteration in range(max_tool_iterations):
            self.logger.info(
                f"Generation iteration {iteration + 1}/{max_tool_iterations}"
            )

            try:
                # Generate text response
                response = await self._generate_text(
                    current_instruction, current_input, generation_params, use_pipeline
                )

                conversation_history.append(
                    {
                        "iteration": iteration + 1,
                        "instruction": current_instruction,
                        "input": current_input,
                        "response": response,
                    }
                )

                # Parse tool calls from response
                tool_calls = self.mcp_client.parse_tool_calls(response)

                if not tool_calls:
                    # No tool calls found, return final response
                    return {
                        "final_response": response,
                        "tool_calls_made": sum(
                            len(h.get("tool_results", [])) for h in conversation_history
                        ),
                        "iterations": iteration + 1,
                        "conversation_history": conversation_history,
                        "success": True,
                    }

                # Execute tool calls
                self.logger.info(f"Executing {len(tool_calls)} tool calls")
                tool_results = await self.mcp_client.execute_tool_calls(tool_calls)

                conversation_history[-1]["tool_calls"] = [
                    {"name": call.name, "parameters": call.parameters}
                    for call in tool_calls
                ]
                conversation_history[-1]["tool_results"] = tool_results

                # Check if any tool calls failed
                failed_calls = [
                    result
                    for result in tool_results
                    if not result.get("success", False)
                ]

                if failed_calls:
                    error_msg = (
                        f"Tool execution failed: {[f['error'] for f in failed_calls]}"
                    )
                    self.logger.error(error_msg)

                    # Generate error response
                    error_response = await self._generate_text(
                        error_msg, current_input, generation_params, use_pipeline
                    )

                    return {
                        "final_response": error_response,
                        "tool_calls_made": len(tool_calls),
                        "iterations": iteration + 1,
                        "conversation_history": conversation_history,
                        "success": False,
                        "errors": failed_calls,
                    }

                # Prepare next iteration with tool results
                tool_results_text = self._format_tool_results(tool_results)
                current_instruction = (
                    f"Based on the following tool results, provide a comprehensive response:\n\n"
                    f"{tool_results_text}\n\nOriginal request: {instruction}"
                )
                current_input = ""

            except Exception as e:
                self.logger.error(
                    f"Error in generation iteration {iteration + 1}: {str(e)}"
                )
                return {
                    "final_response": f"An error occurred during generation: {str(e)}",
                    "tool_calls_made": 0,
                    "iterations": iteration + 1,
                    "conversation_history": conversation_history,
                    "success": False,
                    "errors": [str(e)],
                }

        # Max iterations reached
        final_response = (
            conversation_history[-1]["response"]
            if conversation_history
            else "Maximum iterations reached without completion."
        )

        return {
            "final_response": final_response,
            "tool_calls_made": sum(
                len(h.get("tool_results", [])) for h in conversation_history
            ),
            "iterations": max_tool_iterations,
            "conversation_history": conversation_history,
            "success": False,
            "message": "Maximum tool iterations reached",
        }

    async def _generate_text(
        self,
        instruction: str,
        input_text: str,
        generation_params: GenerationParams,
        use_pipeline: bool = False,
    ) -> str:
        """
        Generate text using the model.

        Args:
            instruction: The instruction/query
            input_text: Additional input context
            generation_params: Generation parameters
            use_pipeline: Whether to use the transformers pipeline for generation

        Returns:
            Generated response text
        """

        if not self._initialized():
            raise RuntimeError("Model and/or tokenizer not initialized")

        prompt = self.format_prompt(instruction, input_text)

        if use_pipeline:
            generation_args = {
                "max_new_tokens": generation_params.max_new_tokens,
                "temperature": generation_params.temperature,
                "top_p": generation_params.top_p,
                "do_sample": generation_params.do_sample,
                "repetition_penalty": generation_params.repetition_penalty,
                "return_full_text": False,  # Only return generated text, not the prompt
                "pad_token_id": generation_params.pad_token_id
                or self.tokenizer.pad_token_id,
                "eos_token_id": self.tokenizer.eos_token_id,
            }

            # Add top_k if specified
            if generation_params.top_k is not None:
                generation_args["top_k"] = generation_params.top_k

            try:
                output = self.pipeline(prompt, **generation_args)
                response = output[0]["generated_text"].strip()
                return response

            except Exception as e:
                self.logger.error(f"Pipeline generation failed: {str(e)}")
                raise

        else:
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    inputs.input_ids,
                    max_new_tokens=generation_params.max_new_tokens,
                    do_sample=generation_params.do_sample,
                    temperature=generation_params.temperature,
                    top_p=generation_params.top_p,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )

            generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Extract only the response part
            if prompt in generated_text:
                return generated_text[len(prompt) :].strip()
            else:
                return generated_text.strip()

    def _format_tool_results(self, tool_results: list[dict[str, Any]]) -> str:
        """
        Format tool results for inclusion in next iteration.

        Args:
            tool_results: list of tool execution results

        Returns:
            Formatted tool results string
        """
        formatted_results = []

        for result in tool_results:
            if result.get("success", False):
                tool_name = result.get("tool", "unknown")
                tool_result = result.get("result", {})

                formatted_results.append(f"Tool: {tool_name}")
                formatted_results.append(f"Result: {tool_result}")
                formatted_results.append("")

        return "\n".join(formatted_results)

    async def batch_generate_async(
        self,
        instructions: list[str],
        input_texts: Optional[list[str]] = None,
        generation_params: Optional[GenerationParams] = None,
        max_tool_iterations: int = 3,
    ) -> list[dict[str, Any]]:
        """
        Generate responses for multiple instructions asynchronously.

        Args:
            instructions: list of instructions/queries
            input_texts: Optional list of input texts
            generation_params: Generation parameters
            max_tool_iterations: Maximum tool iterations per request

        Returns:
            list of response dictionaries
        """
        if input_texts is None:
            input_texts = [""] * len(instructions)

        if len(instructions) != len(input_texts):
            raise ValueError("Instructions and input_texts must have the same length")

        # Process sequentially for now
        # In production, you might want to implement true parallel processing
        results = []
        for i, (instruction, input_text) in enumerate(zip(instructions, input_texts)):
            self.logger.info(f"Processing batch item {i + 1}/{len(instructions)}")
            try:
                result = await self.generate_response(
                    instruction, input_text, generation_params, max_tool_iterations
                )
                results.append(result)
            except Exception as e:
                self.logger.error(f"Batch item {i + 1} failed: {str(e)}")
                results.append(
                    {
                        "final_response": f"Batch processing failed: {str(e)}",
                        "tool_calls_made": 0,
                        "iterations": 0,
                        "conversation_history": [],
                        "success": False,
                        "errors": [str(e)],
                    }
                )

        return results

    def batch_generate(
        self,
        instructions: list[str],
        input_texts: Optional[list[str]] = None,
        generation_params: Optional[GenerationParams] = None,
        max_tool_iterations: int = 3,
    ) -> list[dict[str, Any]]:
        """
        Synchronous wrapper for batch generation.

        Args:
            instructions: list of instructions/queries
            input_texts: Optional list of input texts
            generation_params: Generation parameters
            max_tool_iterations: Maximum tool iterations per request

        Returns:
            list of response dictionaries
        """
        return asyncio.run(
            self.batch_generate_async(
                instructions, input_texts, generation_params, max_tool_iterations
            )
        )

    def get_model_info(self) -> dict[str, Any]:
        """
        Get comprehensive information about the loaded model.

        Returns:
            dictionary containing model information
        """
        info = {
            "base_model_path": self.base_model_path,
            "adapter_path": self.adapter_path,
            "device": str(self.model.device) if self.model else None,
            "model_dtype": str(self.model.dtype) if self.model else None,
            "torch_dtype": str(self.torch_dtype),
            "vocab_size": len(self.tokenizer) if self.tokenizer else None,
            "pad_token_id": self.tokenizer.pad_token_id if self.tokenizer else None,
            "eos_token_id": self.tokenizer.eos_token_id if self.tokenizer else None,
            "model_loaded": self.model is not None,
            "tokenizer_loaded": self.tokenizer is not None,
            "padding_side": self.padding_side,
        }

        # Add available tools if MCP client is available
        try:
            info["available_tools"] = list(self.mcp_client.available_tools.keys())
            info["tools_count"] = len(self.mcp_client.available_tools)
        except Exception as e:
            self.logger.warning(f"Could not get available tools: {e}")
            info["available_tools"] = []
            info["tools_count"] = 0

        # Add model config if available
        if self.model and hasattr(self.model, "config"):
            try:
                config = self.model.config
                info["model_config"] = {
                    "hidden_size": getattr(config, "hidden_size", None),
                    "num_attention_heads": getattr(config, "num_attention_heads", None),
                    "num_hidden_layers": getattr(config, "num_hidden_layers", None),
                    "max_position_embeddings": getattr(
                        config, "max_position_embeddings", None
                    ),
                    "model_type": getattr(config, "model_type", None),
                }
            except Exception as e:
                self.logger.warning(f"Could not get model config: {e}")

        return info

    def __repr__(self) -> str:
        return (
            f"AgentModelHandler("
            f"base_model='{self.base_model_path}', "
            f"adapter='{self.adapter_path}', "
            f"device='{self.device}')"
        )


def setup_model_handler(
    model_info_file: str, base_model: Optional[str] = None
) -> AgentModelHandler:
    """
    Setup the model handler from model info file.

    Model file should contain an object with keys:
    {
        "base_model": "microsoft/Phi-3.5-mini-instruct",
        "adapter": null,
        "dtype": "bfloat16",
        "device": "auto",
        "tokenizer": "/path/to/tokenizer.json",
        "model-dir": "/path/to/model",
        "model-file": "/path/to/model/file"
    }

    Args:
        model_info_file: Path to the model info JSON file (model.json)
        base_model: Optional base model path override

    Returns:
        AgentModelHandler instance
    """
    with open(model_info_file, "r") as f:
        model_info: dict = json.load(f)

    logger.info(f"Load model info from {model_info_file}: {model_info}")
    logger.info("Initializing model...")
    try:
        mcp_server = MCPServerConfig(
            name="slm-mcp-server",
            base_url=model_info.get("mcp-server", "http://localhost:9000/mcp"),
            description="MCP server for tool integration",
        )
        return AgentModelHandler(
            base_model_path=base_model or model_info.get("base_model"),
            mcp_client=MCPClient(servers=[mcp_server]),
            adapter_path=model_info.get("adapter"),
            device=model_info.get("device"),
            torch_dtype=model_info.get("dtype"),
        )
    except Exception as e:
        logger.error(f"Failed to setup model handler: {e}")
        raise


# TODO: add Flask app to serve model over specified endpoints so it can interact over the network

# if False: # __main__ block disabled for notebook
if __name__ == "__main__":
    from model_configs import ModelConfig

    model = setup_model_handler(ModelConfig.MODEL_PATH)

    asyncio.run(model.chat())


## 7. Execution & Testing
Run the dataset builder, trainer, and test inference directly in the notebook.

In [ ]:
import nest_asyncio
import random
from datasets import Dataset
nest_asyncio.apply()

# 1. GENERATE DATASET
print("Generating dataset...")
builder = AgenticDatasetBuilder()
raw_data = builder.generate_dataset(100) # Small subset for demo
random.shuffle(raw_data)
split_idx = int(len(raw_data) * 0.9)
train_dataset = Dataset.from_list([builder._to_dict(ex) if hasattr(ex, '__dict__') else ex for ex in raw_data[:split_idx]])
eval_dataset = Dataset.from_list([builder._to_dict(ex) if hasattr(ex, '__dict__') else ex for ex in raw_data[split_idx:]])
print(f"Generated {len(train_dataset)} training examples and {len(eval_dataset)} eval examples.")

# 2. RUN TRAINER (Demo mode: overrides config to train in-memory using our dataset)
print("\nStarting Trainer Setup...")
cfg_path = "config/training_config.yaml"

# Create a dummy config if it doesn't exist to prevent crash
if not os.path.exists("config"):
    os.makedirs("config")
if not os.path.exists(cfg_path):
    with open(cfg_path, "w") as f:
        f.write("""
model:
  name: microsoft/Phi-3.5-mini-instruct
quantization:
  load_in_4bit: true
lora:
  r: 16
  lora_alpha: 32
  target_modules: ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
  lora_dropout: 0.05
training:
  output_dir: ./results_notebook
  num_train_epochs: 1
  per_device_train_batch_size: 2
  per_device_eval_batch_size: 2
  gradient_accumulation_steps: 4
  learning_rate: 2e-4
  max_steps: 10
  save_steps: 10
  bf16: false
  fp16: true
  pad_to_multiple_of: 8
data:
  max_seq_length: 512
  train_dataset: ""
  eval_dataset: ""
""")

trainer_obj = AgentTrainer(cfg_path)
trainer_obj.setup_model_and_tokenizer()

# Override datasets with our in-memory datasets
trainer_obj.train_dataset = train_dataset.map(
    trainer_obj._tokenize_batch, batched=True, batch_size=32, remove_columns=train_dataset.column_names
)
trainer_obj.eval_dataset = eval_dataset.map(
    trainer_obj._tokenize_batch, batched=True, batch_size=32, remove_columns=eval_dataset.column_names
)

# Train the model
print("\nRunning Training (demo max_steps=10)...")
args = trainer_obj.setup_training_args()
collator = AgentDataCollator(trainer_obj.tokenizer, pad_to_multiple_of=8)

from transformers import Trainer
hf_trainer = Trainer(
    model=trainer_obj.model,
    args=args,
    train_dataset=trainer_obj.train_dataset,
    eval_dataset=trainer_obj.eval_dataset,
    processing_class=trainer_obj.tokenizer,
    data_collator=collator,
    callbacks=[MetricsLoggingCallback()],
)
hf_trainer.train()

# 3. TEST INFERENCE (Model Handler)
print("\nTesting Inference via ModelHandler...")

class MockMCPClient:
    def __init__(self):
        self.available_tools = builder.available_tools

    def parse_tool_calls(self, response: str) -> list:
        import re
        class ToolCall:
            def __init__(self, name, params):
                self.name = name
                self.parameters = params
                self.call_id = None

        calls = []
        pattern = r'<tool_use>\s*<tool_name>(.*?)</tool_name>\s*<parameters>\s*(.*?)\s*</parameters>\s*</tool_use>'
        matches = re.finditer(pattern, response, re.DOTALL)
        for match in matches:
            try:
                calls.append(ToolCall(match.group(1).strip(), json.loads(match.group(2).strip())))
            except Exception:
                pass
        return calls

    async def execute_tool_calls(self, tool_calls: list) -> list:
        results = []
        for call in tool_calls:
            results.append({
                "tool": call.name,
                "result": f"MOCKED RESULT: Successfully executed {call.name}",
                "success": True
            })
        return results

handler = AgentModelHandler(
    base_model_path="microsoft/Phi-3.5-mini-instruct",
    mcp_client=MockMCPClient(),
    device=DEVICE,
)

# Replace with fine-tuned model
handler.model = trainer_obj.model
handler.tokenizer = trainer_obj.tokenizer
handler.sys_prompt = builder._get_system_prompt()

async def test_run():
    print("="*80)
    print("INFERENCE TEST: Single Step Task")
    res1 = await handler.generate_response("summarize the latest advancements in quantum computing", max_tool_iterations=2)
    print(json.dumps(res1, indent=2))

    print("\n" + "="*80)
    print("INFERENCE TEST: Forcing a Tool Result Context")
    tool_results = handler._format_tool_results([{"tool": "web_search", "result": "Quantum computers have reached 1000 qubits.", "success": True}])
    prompt = f"Based on the following tool results, provide a comprehensive response:\n\n{tool_results}\n\nOriginal request: Help me with this task: evaluate quantum capabilities"
    res2 = await handler._generate_text(prompt, "", handler.generation_config, use_pipeline=False)
    print(f"Agent Final Answer:\n{res2}")

# Run async inference
import asyncio
loop = asyncio.get_event_loop()
loop.run_until_complete(test_run())
